In [25]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

In [26]:
df = pd.read_csv("../1-Data/heart.csv")
print('Shape',df.shape)

Shape (918, 12)


1. Remove the RestingBP outliers
2. Split train/test
3. Change the cholesterol = 0
4. Encode categorical

In [27]:
df_model = df.copy()
# 1. Remove RestingBP outliers
df_model = df_model[df_model["RestingBP"] > 0]

print("Shape after removing RestingBP outliers:")
print(df_model.shape)


Shape after removing RestingBP outliers:
(917, 12)


In [28]:
# 2. Split the dataset
df_train, df_test = train_test_split(
    df_model,
    test_size=0.2,
    random_state=42,
    stratify=df_model['HeartDisease']
)

df_train.to_csv("../1-Data/train_data.csv", index = False)
df_test.to_csv("../1-Data/test_data.csv", index=False)

In [29]:
df = pd.read_csv("../1-Data/train_data.csv")
print("Shape:", df.shape)
df_model = df.copy()

Shape: (733, 12)


In [30]:
# 3. Change the cholesterol = 0 
df_model["Cholesterol_missing"] = (df_model["Cholesterol"] == 0).astype(int)

# Replace 0 with NaN
df_model["Cholesterol"] = (df_model["Cholesterol"].replace(0, np.nan))

# Fill missing values using median
median_chol = df_model["Cholesterol"].median()

df_model["Cholesterol"] = (df_model["Cholesterol"].fillna(median_chol))

print("\nMissing values after imputation:")
print(df_model["Cholesterol"].isnull().sum())


Missing values after imputation:
0


In [31]:
# 4. Encode categorical
df_model["Sex"] = (df_model["Sex"] == "M").astype(int)

df_model["ExerciseAngina"] = (df_model["ExerciseAngina"] == "Y").astype(int)

# Ordinal encoding
df_model["ST_Slope"] = df_model["ST_Slope"].map({
    "Down": 0,
    "Flat": 1,
    "Up": 2
})

# One-hot encoding
df_model = pd.get_dummies(
    df_model,
    columns=["ChestPainType", "RestingECG"],
    drop_first=True,
    dtype=int 
)

print("\nShape after encoding:")
print(df_model.shape)


Shape after encoding:
(733, 16)


In [32]:
df_model.to_csv("../1-Data/train_processed.csv",index=False)

In [33]:
df = pd.read_csv("../1-Data/train_processed.csv")
print('Shape',df.head())

Shape    Age  Sex  RestingBP  Cholesterol  FastingBS  MaxHR  ExerciseAngina  \
0   49    1        128        212.0          0     96               1   
1   68    1        134        254.0          1    151               1   
2   59    1        126        218.0          1    134               0   
3   67    1        125        254.0          1    163               0   
4   52    1        160        331.0          0     94               1   

   Oldpeak  ST_Slope  HeartDisease  Cholesterol_missing  ChestPainType_ATA  \
0      0.0         1             1                    0                  0   
1      0.0         2             0                    0                  0   
2      2.2         1             1                    0                  0   
3      0.2         1             1                    0                  0   
4      2.5         1             1                    0                  0   

   ChestPainType_NAP  ChestPainType_TA  RestingECG_Normal  RestingECG_ST  
0          

# Explanation of the encoding part
Change sex Male -> 1, Female -> 0  
Change ExerciseAngina to Y -> 1 and N = 0  
Change Down -> 0, Flat -> 1, Up -> 2 of ST_Slope  
Because the chestpaintyype and restingECG are important and cannot classify and classify as 0,1,2 (NO natural order) so split them to smaller category and also, 0 means false, 1 means true. 